# Inside the agent's sandbox

Notebook 01 used sandboxes directly. Here a **Managed Deep Agent** uses one. We ask the Snowflake analyst agent (`../mda-snowflake-analyst/`) two questions in the same conversation, then open its sandbox to see what it actually did.

What to watch for:

- The agent writes and runs its own analysis code in the sandbox, and hands back a chart and a shareable dashboard.
- The sandbox belongs to the **conversation**: the follow-up question reuses the data already there, with no new Snowflake queries.
- Everything the agent produced is inspectable afterwards: data, code, outputs.

**Setup**

1. The agent running locally: `cd ../mda-snowflake-analyst && mda dev` (see that folder's README).
2. `LANGSMITH_API_KEY` in `../mda-snowflake-analyst/.env` set to a **personal access token**, so the agent queries Snowflake as you.

In [1]:
import json
import os
import time

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langgraph_sdk import get_client
from langsmith.sandbox import SandboxClient

load_dotenv("../mda-snowflake-analyst/.env")
agent = get_client(url="http://localhost:2024", api_key=os.environ["LANGSMITH_API_KEY"])
sandboxes = SandboxClient()
AGENT_ID = "snowflake-analyst"


async def ask(thread_id: str, question: str) -> list[str]:
    """Send one message, print each tool call as it happens, show the answer. Returns the tool names used."""
    start, used = time.time(), []
    async for chunk in agent.runs.stream(
        thread_id, AGENT_ID, input={"messages": [{"role": "user", "content": question}]}, stream_mode="updates"
    ):
        for update in (chunk.data or {}).values() if chunk.event == "updates" else []:
            for message in (update or {}).get("messages", []) if isinstance(update, dict) else []:
                for call in message.get("tool_calls") or []:
                    used.append(call["name"])
                    args = json.dumps(call["args"])
                    print(f"[{time.time() - start:5.1f}s] {call['name']:<26} {args[:90]}{'…' if len(args) > 90 else ''}")
    state = await agent.threads.get_state(thread_id)
    answer = state["values"]["messages"][-1]["content"]
    if not isinstance(answer, str):
        answer = "".join(block.get("text", "") for block in answer if isinstance(block, dict))
    print(f"[{time.time() - start:5.1f}s] done")
    display(Markdown(answer))
    return used

## Turn 1: build a dashboard

Watch the tool calls: the agent loads its `dashboard` skill, asks Snowflake (through LangSmith Tools) for a few small aggregated results, writes the rows and a Python script into its sandbox in one batch, runs it, and shares the outputs.

In [2]:
thread = await agent.threads.create()
thread_id = thread["thread_id"]

turn1 = await ask(
    thread_id,
    "Build me a revenue dashboard: monthly revenue trend, revenue by region by year, and the top-line KPIs.",
)

[  7.6s] read_file                  {"file_path": "/skills/dashboard/SKILL.md", "limit": 1000}
[  7.6s] snowflake_sales_analyst    {"message": "Total revenue by month (all available months)"}
[  7.6s] snowflake_sales_analyst    {"message": "Total revenue by region by year"}


[ 16.5s] snowflake_run_sql          {"sql": "SELECT *\nFROM SEMANTIC_VIEW(\n    FLEET_MCP.SEMANTIC.SALES_SV\n    DIMENSIONS or…
[ 16.5s] snowflake_run_sql          {"sql": "SELECT *\nFROM SEMANTIC_VIEW(\n    FLEET_MCP.SEMANTIC.SALES_SV\n    METRICS total…


[ 51.6s] write_file                 {"file_path": "/workspace/data/monthly_revenue.csv", "content": "month,revenue\n1992-01-01…
[ 51.6s] write_file                 {"file_path": "/workspace/data/region_year.csv", "content": "region,year,revenue\nEUROPE,1…
[ 51.6s] write_file                 {"file_path": "/workspace/dashboard.py", "content": "import pandas as pd, altair as alt, v…


[ 59.7s] execute                    {"command": "mkdir -p /workspace/out && python3 /workspace/dashboard.py"}


[ 69.0s] share_file                 {"path": "/workspace/out/preview_v1.png"}
[ 69.0s] share_file                 {"path": "/workspace/out/dashboard_v1.html"}


[ 76.5s] done


Revenue totals **$218.1B** from Jan 1992 to early Aug 1998, and growth is flat.

- **Total revenue:** $218.1B. Both the monthly and the region-by-year queries sum to the same figure.
- **Average full month:** $2.76B.
- **1998 vs 1997 (Jan–Jul):** −0.01%. Full-year 1997 was −0.7% vs 1996.
- **Top region:** Europe at $44.0B, followed by Asia ($43.9B), America ($43.6B), Africa ($43.5B) and Middle East ($43.2B). The gap from first to last is under 2%.
- **Seasonality:** February is the low point every year, at about $2.5–2.6B against roughly $2.8B in other months.

**Caveat:** the data ends in early Aug 1998. August 1998 ($0.17B) is a partial month, so I left it off the trend line and the averages. 1998 is a partial year, so the yearly region chart shows 1992–1997. The 1998 region figures are in a side table.

![Chart](https://<sandbox>.smithbox.dev/<signed-token>)

[Open the dashboard](https://<sandbox>.smithbox.dev/<signed-token>)

## Open the agent's sandbox

MDA gives each conversation thread its own sandbox. The agent's sharing tools record which sandbox a file came from, so we can find it from the thread.

In [3]:
state = await agent.threads.get_state(thread_id)
sandbox_name = next(
    m["artifact"]["sandbox"]
    for m in state["values"]["messages"]
    if m.get("type") == "tool" and isinstance(m.get("artifact"), dict) and "sandbox" in m["artifact"]
)
sb = sandboxes.get_sandbox(sandbox_name)
print("sandbox:", sb.name, "| status:", sb.status)
print(f"idle stop after {sb.idle_ttl_seconds // 60} min, deleted {sb.delete_after_stop_seconds // 86400} days after stopping")
print()
print(sb.run("cd /workspace && find . -type f | sort").stdout)

sandbox: dev-a0d11a8994177deb--d5ffecef0867 | status: ready
idle stop after 30 min, deleted 14 days after stopping



./dashboard.py
./data/monthly_revenue.csv
./data/region_year.csv
./out/dashboard_v1.html
./out/preview_v1.png



The data the agent pulled from Snowflake, as it saved it:

In [4]:
print(sb.run("cd /workspace/data && wc -l *.csv && echo && head -5 monthly_revenue.csv").stdout)

  81 monthly_revenue.csv
  36 region_year.csv
 117 total

month,revenue
1992-01-01,2812229777.73
1992-02-01,2618065218.41
1992-03-01,2804113927.98
1992-04-01,2702383581.75



The analysis code the agent wrote and ran (first 40 lines):

In [5]:
print(sb.run("head -40 /workspace/dashboard.py").stdout)

import pandas as pd, altair as alt, vl_convert as vlc

monthly = pd.read_csv("/workspace/data/monthly_revenue.csv", parse_dates=["month"])
reg = pd.read_csv("/workspace/data/region_year.csv")
print(f"monthly_revenue: {len(monthly)} rows, {monthly.month.min():%Y-%m} to {monthly.month.max():%Y-%m}")
print(f"region_year: {len(reg)} rows, {reg.region.nunique()} regions, years {reg.year.min()}-{reg.year.max()}")
assert len(monthly) == 80 and len(reg) == 35

# consistency check: monthly totals vs region totals
print(f"total monthly={monthly.revenue.sum():,.0f} region={reg.revenue.sum():,.0f}")

# Aug 1998 is a partial month (data ends early Aug) -> exclude from trend/averages
full = monthly[monthly.month < "1998-08-01"].copy()
reg["label"] = reg.year.astype(str).where(reg.year != 1998, "1998 (Jan-Jul+)")
yr = reg.groupby("year").revenue.sum()

total = monthly.revenue.sum()
ytd98 = full[full.month.dt.year == 1998].revenue.sum()
ytd97 = full[(full.month.dt.year == 1997) & (full.month.dt.month 

Where did Altair and statsmodels come from? `sandbox/setup.sh` (one `pip install` line) ran **once**, when the agent was built, and was captured as a snapshot. Every new conversation's sandbox is a clone of it, so nothing is installed at run time.

In [6]:
print(sb.run("pip list 2>/dev/null | grep -i -E '^(pandas|altair|vl-convert-python|statsmodels) '").stdout)

altair                    6.3.0
pandas                    3.1.0
statsmodels               0.15.0
vl-convert-python         1.9.0.post1



## Turn 2: add a forecast

Same conversation, so the same sandbox. The agent should reuse the data and script it already saved, train a forecasting model, check it against the last 6 known months, and report how it did, without going back to Snowflake.

In [7]:
turn2 = await ask(thread_id, "Add a 6-month revenue forecast to the dashboard.")

snowflake_calls = [name for name in turn2 if name.startswith("snowflake_")]
print(f"\nSnowflake calls in turn 1: {sum(n.startswith('snowflake_') for n in turn1)}, in turn 2: {len(snowflake_calls)}")

[  7.5s] read_file                  {"file_path": "/workspace/dashboard.py", "limit": 200}


[ 22.3s] edit_file                  {"file_path": "/workspace/dashboard.py", "old_string": "BILLIONS = alt.Axis(", "new_string…
[ 22.3s] read_file                  {"file_path": "/workspace/dashboard.py", "offset": 60, "limit": 10}


[ 33.9s] edit_file                  {"file_path": "/workspace/dashboard.py", "old_string": "trend = line.properties(width=980,…
[ 33.9s] edit_file                  {"file_path": "/workspace/dashboard.py", "old_string": "    (\"Avg / full month\", f\"${av…
[ 33.9s] edit_file                  {"file_path": "/workspace/dashboard.py", "old_string": ".kpis{{display:grid;grid-template-…
[ 33.9s] edit_file                  {"file_path": "/workspace/dashboard.py", "old_string": "Regions are within ~1% of each oth…
[ 33.9s] edit_file                  {"file_path": "/workspace/dashboard.py", "old_string": "preview_v1.png", "new_string": "pr…


[ 40.0s] edit_file                  {"file_path": "/workspace/dashboard.py", "old_string": "trend = line.properties(width=980,…


[ 47.8s] edit_file                  {"file_path": "/workspace/dashboard.py", "old_string": "    (\"Avg / full month\", f\"${av…
[ 47.8s] execute                    {"command": "sleep 1; grep -n \"band = alt\" /workspace/dashboard.py"}


...

[ 69.7s] execute                    {"command": "python3 /workspace/dashboard.py"}


[ 78.9s] share_file                 {"path": "/workspace/out/preview_v2.png"}
[ 78.9s] share_file                 {"path": "/workspace/out/dashboard_v2.html"}


[ 87.1s] done


The forecast puts revenue for Aug 1998 – Jan 1999 at **$16.8B**, which is +0.9% on the same six months a year earlier.

| Month | Forecast |
|---|---|
| Aug 1998 | $2.82B |
| Sep 1998 | $2.73B |
| Oct 1998 | $2.81B |
| Nov 1998 | $2.74B |
| Dec 1998 | $2.82B |
| Jan 1999 | $2.84B |

The dashboard now has a dashed forecast line with a 95% band on the trend chart. It also has two new KPI cards, one for the 6-month forecast and one for backtest error, plus a "Forecast model" card.

![Chart](https://<sandbox>.smithbox.dev/<signed-token>)

[Open the dashboard](https://<sandbox>.smithbox.dev/<signed-token>)

**How I built the forecast**
- **Training data:** the 79 complete months from Jan 1992 to Jul 1998. I dropped the partial Aug 1998 month.
- **Method:** Holt-Winters exponential smoothing, with an additive trend and additive 12-month seasonality.
- **Backtest:** I trained on Jan 1992 – Jan 1998 (73 months) and predicted Feb – Jul 1998, which I already had actuals for. It was off by 0.8% on average. Simply repeating the same month from last year was off by 1.0%, so the model is only slightly better than that baseline.
- **Band:** the forecast ±1.96 × the standard deviation of the model's fit errors ($27M). It's narrow because this series is very regular.
- **Limits:**
  - Revenue has been flat for six years, with a February dip, so the model mostly repeats that pattern.
  - It can't anticipate price changes, demand shifts or other outside events.
  - The band ignores uncertainty in the fitted parameters, so treat it as a rough range.

I kept v1 of the dashboard unchanged.


Snowflake calls in turn 1: 4, in turn 2: 0


The modeling the agent did, straight from its script's output (it reruns in a few seconds):

In [8]:
print(sb.run("cd /workspace && python3 dashboard.py | grep -i -E 'backtest|model|forecast'").stdout)

backtest: trained on 73 months (1992-01 to 1998-01), predicted the last 6 (1998-02 to 1998-07), MAPE = 0.8% (same-month-last-year baseline: 1.0%)
model: Holt-Winters on 79 months, alpha=0.01 beta=0.00 gamma=0.00
forecast: Aug 1998 $2.82B, Sep 1998 $2.73B, Oct 1998 $2.81B, Nov 1998 $2.74B, Dec 1998 $2.82B, Jan 1999 $2.84B



Both versions are kept. Shared links point at a specific file, so the agent writes a new version instead of overwriting:

In [9]:
print(sb.run("ls -la /workspace/out").stdout)

total 1332
drwxr-xr-x 2 root root   4096 Oct 26 13:59 .
drwxr-xr-x 4 root root   4096 Oct 26 13:58 ..
-rw-r--r-- 1 root root  36538 Oct 26 13:58 dashboard_v1.html
-rw-r--r-- 1 root root  38740 Oct 26 14:00 dashboard_v2.html
-rw-r--r-- 1 root root 682098 Oct 26 13:58 preview_v1.png
-rw-r--r-- 1 root root 592792 Oct 26 14:00 preview_v2.png



## What this showed

- **Code beyond SQL.** Snowflake answered *what happened*; the sandbox let the agent build a dashboard and fit a forecast.
- **State per conversation.** Turn 2 reused turn 1's data and script, and left an inspectable record of what the agent did.
- **No credentials in the sandbox.** Snowflake access went through LangSmith Tools as you; the sandbox only ever held data and code.
- **Shareable, static outputs.** The dashboard is one HTML file with charts rendered to SVG inside the sandbox, shared with a 7-day download link that keeps working after the sandbox stops.
- **Scratch space with a lifecycle.** The sandbox stops when idle and is deleted 14 days after stopping.

How the project is put together, and how to deploy it: `../mda-snowflake-analyst/README.md` and `../docs/mda-workflow.md`.